# Software Development Lifecycle using Lang-Graph

In [1]:
import sys
print(sys.executable)

c:\Github\SDLC-using-Lang-Graph\env\Scripts\python.exe


In [2]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [3]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model = "gpt-4.1-mini",
    temperature = 0.1,
)

In [7]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langgraph.types import Command, interrupt
import datetime

In [8]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
config = {"configurable": {"thread_id" : "1"}}

In [9]:
import sys
from pathlib import Path

# Go up one directory from "testing grounds" to the project root
project_root = Path.cwd().parent

sys.path.append(str(project_root))

In [ ]:
from tools.planner import create, append, rewrite, read_plan, summarize_plan, update_plan
from tools.designer import create, append, rewrite, read_design
import tools

In [ ]:
plan_tools = [tools.planner.create, tools.planner.append, tools.planner.rewrite, tools.planner.read_plan, tools.planner.summarize_plan, tools.planner.update_plan]
implementor_tools = [tools.coder.create_file, tools.coder.update_file, tools.coder.read_workspace, tools.coder.read_file]
design_tools = [tools.designer.create, tools.designer.append, tools.designer.rewrite, tools.designer.read_design]
testing_tools = [tools.tester.read_workspace, tools.tester.critique_file]

In [ ]:
planner = llm.bind_tools(tools = plan_tools)

def planning_agent(state:State):
    response = planner.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
designer = llm.bind_tools(tools = plan_tools)

def designing_agent(state:State):
    response = designer.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tester = llm.bind_tools(tools = testing_tools)

@tool
def testing_agent(state:State):
    """
        
    """
    response = tester.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
implementor_tools = [tools.coder.create_file, tools.coder.update_file, tools.coder.read_workspace, tools.coder.read_file, testing_agent]

In [ ]:
implementor = llm.bind_tools(tools = plan_tools)

def implementing_agent(state:State):
    response = implementor.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
planner_builder = StateGraph(State)

planner_builder.add_node("Planner", planning_agent)
planner_builder.add_node("tools", ToolNode(tools=plan_tools))

planner_builder.add_edge(START, "Planner")
planner_builder.add_conditional_edges(
    "Planner",
    tools_condition
)
planner_builder.add_edge("tools", "Planner")

graph = planner_builder.compile(checkpointer=memory)